##### Input : GRD(dB).tif, AOI.shp, DEM.tif, slope.tif
##### 임계값 : 수체 탐지할 dB 값, 경사도

In [ ]:
# 어떤 tif 파일이든 shp 영역에 맞게 잘라줌

import os
import rasterio
from rasterio.mask import mask
import geopandas as gpd

# ---------- 사용자 입력 ----------
tif_list = [
    r"<DATA_ROOT>\07_disaster\flood\Korea\202507_Jinju\process\S1A_IW_GRDH_1SDV_20250719_Orb_NR_Cal_Spk_TC_dB.tif",
    r"<DATA_ROOT>\07_disaster\flood\Korea\202507_Jinju\process\S1C_IW_GRDH_1SDV_20250712_Orb_NR_Cal_Spk_TC_dB.tif",
    r"<DATA_ROOT>\07_disaster\flood\Korea\202507_Jinju\process\S1C_IW_GRDH_1SDV_20250718_Orb_NR_Cal_Spk_TC_dB.tif",
    r"<DATA_ROOT>\07_disaster\flood\Korea\202507_Jinju\dem\output_hh.tif",
    # r"<DATA_ROOT>\07_disaster\flood\Korea\202507_Jinju\waterbody\occurrence_120E_40Nv1_4_2021.tif"
]

aoi_path = r"<DATA_ROOT>\07_disaster\flood\Korea\202507_Jinju\shp\AOI.shp"  # AOI shapefile 경로
output_dir = r"<DATA_ROOT>\07_disaster\flood\Korea\202507_Jinju\test"    # 출력 경로
os.makedirs(output_dir, exist_ok=True)

# -----------------------------------

# AOI 읽기
aoi = gpd.read_file(aoi_path)
aoi = aoi.to_crs("EPSG:4326")  # AOI 좌표계를 WGS84로 변환 (tif와 동일해야 함)

# 각 tif 파일 클립 수행
for tif_path in tif_list:
    with rasterio.open(tif_path) as src:
        # 좌표계 맞추기
        if aoi.crs.to_string() != src.crs.to_string():
            aoi = aoi.to_crs(src.crs)

        # AOI 영역으로 마스크
        out_image, out_transform = mask(dataset=src, shapes=aoi.geometry, crop=True)
        out_meta = src.meta.copy()
        out_meta.update({
            "driver": "GTiff",
            "height": out_image.shape[1],
            "width": out_image.shape[2],
            "transform": out_transform
        })

        # 저장 경로
        out_name = os.path.basename(tif_path).replace(".tif", "_clipped.tif")
        out_path = os.path.join(output_dir, out_name)

        # 저장
        with rasterio.open(out_path, "w", **out_meta) as dest:
            dest.write(out_image)

        print(f"✔ 클립 완료: {out_path}")


In [ ]:
# @ 이하의 db tif (수체)에서 @ 이상의 경사를 가지는 부분(물이 고일 수 없음)은 제거하여 이진 tif 생성

import os
import numpy as np
import rasterio
from rasterio.enums import Resampling
from rasterio.warp import reproject, Resampling as WarpResampling
import re

# ---------------- 사용자 입력 ----------------
db_path = r"<DATA_ROOT>\07_disaster\flood\Korea\202507_Jinju\test\S1C_IW_GRDH_1SDV_20250718_Orb_NR_Cal_Spk_TC_dB_clipped.tif"
slope_path = r"<DATA_ROOT>\07_disaster\flood\Korea\202507_Jinju\dem\slope.tif"
output_dir = r"<DATA_ROOT>\07_disaster\flood\Korea\202507_Jinju\test"
os.makedirs(output_dir, exist_ok=True)

# ---------------- 필터 조건 ----------------
SLOPE_THRESHOLD = 15  # 15도 이상인 곳 지우기
DB_THRESHOLD = -15    # -15 db 이하인 곳 지우기

# ---------------- 출력 파일명 생성 ----------------
# 예: S1A_IW_GRDH_1SDV_20250719_Orb_... → S1A, 20250719 추출
basename = os.path.basename(db_path)
match = re.search(r'(S1[ABC])_.*?_(\d{8})_', basename)
if match:
    satellite = match.group(1)
    date = match.group(2)
    out_filename = f"{satellite}_{date}_slopeFiltered_binaryMask.tif"
else:
    out_filename = "slopeFiltered_binaryMask.tif"  # fallback

output_path = os.path.join(output_dir, out_filename)

# ---------------- Sentinel-1 dB 영상 불러오기 ----------------
with rasterio.open(db_path) as db_src:
    db_data = db_src.read(1)
    db_meta = db_src.meta
    db_crs = db_src.crs
    db_transform = db_src.transform
    db_height = db_src.height
    db_width = db_src.width

# ---------------- slope.tif 재투영 및 리샘플 ----------------
with rasterio.open(slope_path) as slope_src:
    slope_data_resampled = np.empty((db_height, db_width), dtype=np.float32)

    reproject(
        source=rasterio.band(slope_src, 1),
        destination=slope_data_resampled,
        src_transform=slope_src.transform,
        src_crs=slope_src.crs,
        dst_transform=db_transform,
        dst_crs=db_crs,
        dst_resolution=(db_transform.a, -db_transform.e),
        resampling=WarpResampling.bilinear
    )

# ---------------- 조건 마스크 생성 ----------------
mask = ((slope_data_resampled < SLOPE_THRESHOLD) & (db_data < DB_THRESHOLD)).astype("uint8")

# ---------------- 저장 ----------------
db_meta.update({
    "dtype": "uint8",
    "nodata": 0,
    "count": 1
})

with rasterio.open(output_path, 'w', **db_meta) as dst:
    dst.write(mask, 1)

print(f"✔ 저장 완료: {output_path}")


In [ ]:
# 두 시기의 이진 tif 를 차분하여 홍수가 발생한 영역만 추출

import rasterio
import numpy as np
import os

# 입력 파일 경로
binary_after_path = r"<DATA_ROOT>\07_disaster\flood\Korea\202507_Jinju\test\S1C_20250718_slopeFiltered_binaryMask.tif"   # 홍수 이후
binary_before_path = r"<DATA_ROOT>\07_disaster\flood\Korea\202507_Jinju\test\S1C_20250712_slopeFiltered_binaryMask.tif" # 홍수 이전
output_path = r"<DATA_ROOT>\07_disaster\flood\Korea\202507_Jinju\test\20250712_20250718_diff_binary.tif"    # 홍수 발생 영역

# 1. 두 마스크 파일 읽기
with rasterio.open(binary_after_path) as after_src, rasterio.open(binary_before_path) as before_src:
    after = after_src.read(1).astype(np.int8)
    before = before_src.read(1).astype(np.int8)
    meta = after_src.meta

# 2. 차분: after - before
diff = after - before

# 3. 홍수 발생한 지역만 추출: 1 (새롭게 생긴 수체), 나머지 0
flood_mask = (diff == 1).astype("uint8")

# 4. 저장 메타 설정
meta.update({
    "dtype": "uint8",
    "count": 1,
    "nodata": 0
})

# 5. 결과 저장
with rasterio.open(output_path, 'w', **meta) as dst:
    dst.write(flood_mask, 1)

print(f"✔ 홍수 발생 영역 추출 완료: {output_path}")


### 기상청 강수량 Data

In [ ]:
# 내가 원하는 기간에 대한 강수량 자료 전부 다운로드

import os
import requests
from datetime import datetime, timedelta

# ───── 사용자 입력 ─────
my_api_key = os.environ.get("KMA_API_KEY", "")
start_time = '202507160000'  # 시작 시간 (예: 2025년 7월 19일 09:00)
end_time   = '202507192350'  # 종료 시간 (예: 2025년 7월 19일 12:00)
data_type = 'bin'
cmp_type = 'hsp'
save_dir = r'<DATA_ROOT>\07_disaster\flood\Korea\202507_Jinju\HSP'
os.makedirs(save_dir, exist_ok=True)

# ───── API 설정 ─────
api_url = 'https://apihub.kma.go.kr/api/typ04/url/rdr_cmp_file.php'

# ───── 시간 반복 ─────
current = datetime.strptime(start_time, '%Y%m%d%H%M')
end = datetime.strptime(end_time, '%Y%m%d%H%M')
interval = timedelta(minutes=10)

while current <= end:
    tm = current.strftime('%Y%m%d%H%M')
    file_name = f'RDR_CMP_{cmp_type.upper()}_PUB_{tm}.bin.gz'
    file_path = os.path.join(save_dir, file_name)

    # 파일이 이미 있는 경우 건너뜀
    if os.path.exists(file_path):
        print(f'이미 존재: {file_name}')
    else:
        print(f'다운로드 시도: {file_name}')
        params = {
            'tm': tm,
            'data': data_type,
            'cmp': cmp_type,
            'authKey': my_api_key
        }
        try:
            response = requests.get(api_url, params=params)
            if response.status_code != 200 or len(response.content) < 500:
                print(f'❌ 실패: {tm} → {response.content.decode("utf-8")}')
            else:
                with open(file_path, 'wb') as f:
                    f.write(response.content)
                print(f'✅ 저장 완료: {file_name}')
        except Exception as e:
            print(f'에러: {tm} → {e}')
    
    # 다음 시간으로 이동
    current += interval


In [ ]:
# openstreetmap 위에 다운받은 강수량 자료 & AOI.shp 파일 오버레이해서 png로 저장

import os
import re
import gzip
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.cm import ScalarMappable
from mpl_toolkits.axes_grid1 import make_axes_locatable
from rasterio.transform import Affine
from rasterio.warp import calculate_default_transform, reproject, Resampling
import contextily as ctx
from pyproj import Transformer
import geopandas as gpd
from matplotlib.ticker import FuncFormatter

# ───── 기본 설정 ─────
nx, ny = 2305, 2881
center_x, center_y = 1121, 1681
resolution = 500
source_crs = "+proj=lcc +lat_1=30 +lat_2=60 +lat_0=38 +lon_0=126 +x_0=0 +y_0=0 +ellps=WGS84 +units=m +no_defs"
dst_crs = "EPSG:3857"
aoi_path = r"<DATA_ROOT>\07_disaster\flood\Korea\202507_Jinju\shp\AOI.shp"
input_dir = r"<DATA_ROOT>\07_disaster\flood\Korea\202507_Jinju\HSP"
output_dir = os.path.join(input_dir, "PNG")
os.makedirs(output_dir, exist_ok=True)

# ───── 시각화 범위 (EPSG:3857) ─────
geo_transformer = Transformer.from_crs("EPSG:4326", dst_crs, always_xy=True)
x_min, y_min = geo_transformer.transform(125.5, 34.0)
x_max, y_max = geo_transformer.transform(130.5, 37.5)

# EPSG:3857 → EPSG:4326 라벨용 변환기
label_transformer = Transformer.from_crs(dst_crs, "EPSG:4326", always_xy=True)

def format_lon(x, pos=None):
    lon, _ = label_transformer.transform(x, y_min)
    return f"{lon:.2f}°E"

def format_lat(y, pos=None):
    _, lat = label_transformer.transform(x_min, y)
    return f"{lat:.2f}°N"

# ───── 색상표 설정 ─────
colormap_rain = ListedColormap(np.array([
    [250, 250, 250], [0, 200, 255], [0, 155, 245], [0, 74, 245],
    [0, 255, 0], [0, 190, 0], [0, 140, 0], [0, 90, 0],
    [255, 255, 0], [255, 220, 31], [249, 205, 0], [224, 185, 0], [204, 170, 0],
    [255, 102, 0], [255, 50, 0], [210, 0, 0], [180, 0, 0],
    [224, 169, 255], [201, 105, 255], [179, 41, 255], [147, 0, 228],
    [179, 180, 222], [76, 78, 177], [0, 3, 144], [51, 51, 51]
]) / 255)
colormap_rain.set_bad([0, 0, 0, 0])
bounds = np.array([
    0, 0.1, 0.5, 1, 2, 3, 4, 5,
    6, 7, 8, 9, 10, 15, 20, 25, 30,
    40, 50, 60, 70, 90, 110, 150
])
norm = BoundaryNorm(boundaries=bounds, ncolors=len(colormap_rain.colors))
ticks = bounds

# ───── AOI 불러오기 ─────
gdf = gpd.read_file(aoi_path).to_crs(dst_crs)


# ───── 파일 반복 처리 ─────
for bin_file in sorted(os.listdir(input_dir)):
    if not bin_file.endswith(".bin.gz"):
        continue

    full_path = os.path.join(input_dir, bin_file)

    try:
        match = re.search(r'(\d{12})', bin_file)
        timestamp = match.group(1) if match else "unknown"

        with open(full_path, 'rb') as f:
            raw = gzip.decompress(f.read())
        rain_rate = np.frombuffer(raw, dtype=np.int16, offset=1024).astype(np.float32).reshape(ny, nx)
        rain_rate[rain_rate <= -30000] = np.nan
        rain_rate /= 100

        normalized = norm(rain_rate)
        rgba = colormap_rain(normalized)
        rgba = (rgba * 255).astype(np.uint8)

        low_rain_mask = (rain_rate >= 0) & (rain_rate < 0.1)
        rgba[low_rain_mask, 3] = 0
        rgba[~low_rain_mask & ~np.isnan(rain_rate), 3] = int(0.4 * 255)
        rgba[np.isnan(rain_rate)] = [0, 0, 0, 0]

        source_transform = Affine.scale(resolution, resolution) * Affine.translation(-center_x, -center_y)
        source_bounds = {
            'left': -center_x * resolution,
            'bottom': (ny - center_y) * resolution,
            'right': (nx - center_x) * resolution,
            'top': -center_y * resolution
        }
        dest_transform, dest_width, dest_height = calculate_default_transform(
            src_crs=source_crs,
            dst_crs=dst_crs,
            width=nx,
            height=ny,
            **source_bounds,
        )
        converted_array = np.zeros((dest_height, dest_width, 4), dtype=np.uint8)
        for i in range(4):
            reproject(
                source=rgba[:, :, i],
                destination=converted_array[:, :, i],
                src_transform=source_transform,
                src_crs=source_crs,
                dst_transform=dest_transform,
                dst_crs=dst_crs,
                resampling=Resampling.nearest,
            )

        fig, ax = plt.subplots(figsize=(10, 10))
        ax.set_xlim(x_min, x_max)
        ax.set_ylim(y_min, y_max)
        ctx.add_basemap(ax, crs=dst_crs, source=ctx.providers.OpenStreetMap.Mapnik, zoom=8)

        left, top = dest_transform * (0, 0)
        right, bottom = dest_transform * (dest_width, dest_height)
        extent = [left, right, bottom, top]
        ax.imshow(converted_array, extent=extent, origin='upper')

        divider = make_axes_locatable(ax)
        cax = divider.append_axes("right", size="5%", pad=0.05)
        sm = ScalarMappable(cmap=colormap_rain, norm=norm)
        sm.set_array([])
        cbar = plt.colorbar(sm, cax=cax, ticks=ticks)
        cbar.ax.set_title("mm/h", fontsize=9)

        gdf.boundary.plot(ax=ax, edgecolor='red', linewidth=2)
        gdf.plot(ax=ax, facecolor='none', edgecolor='red', linewidth=2)

        # 위경도 라벨 표시
        ax.xaxis.set_major_formatter(FuncFormatter(format_lon))
        ax.yaxis.set_major_formatter(FuncFormatter(format_lat))
        ax.set_xlabel("Longitude")
        ax.set_ylabel("Latitude")

        # 날짜/시간 제목 설정
        if timestamp != "unknown":
            time_str = f"{timestamp[:4]}-{timestamp[4:6]}-{timestamp[6:8]} {timestamp[8:10]}:{timestamp[10:12]}"
            ax.set_title(time_str, fontsize=16, color='black')

        save_name = f"{timestamp}_HSP.png"
        save_path = os.path.join(output_dir, save_name)
        plt.savefig(save_path, dpi=300, bbox_inches='tight', transparent=False)
        print(f"✅ 저장 완료: {save_path}")
        plt.close()

    except Exception as e:
        print(f"❌ 오류 발생: {bin_file} → {e}")


In [ ]:
# 모든 기간의 png 파일 이용해서 gif 만들기

import imageio
import os

# PNG 파일들이 있는 폴더
input_dir = r"<DATA_ROOT>\07_disaster\flood\Korea\202507_Jinju\HSP\PNG"
output_gif = r"<DATA_ROOT>\07_disaster\flood\Korea\202507_Jinju\HSP\PNG\rain_animation_all.gif"

# 프레임 간 시간 (초 단위)
frame_duration = 1

# PNG 파일 목록 가져오기 (정렬)
png_files = sorted([f for f in os.listdir(input_dir) if f.endswith('.png')])

# 전체 경로로 변환
image_paths = [os.path.join(input_dir, f) for f in png_files]

# 이미지 읽고 GIF로 저장
images = [imageio.v3.imread(path) for path in image_paths]
imageio.mimsave(output_gif, images, duration=frame_duration)

print(f"✅ GIF 저장 완료: {output_gif}")
